# Polymarket Smart Money Tracker

This notebook fetches leaderboard data across all Polymarket categories, identifies consistently profitable wallets, and exports the data to a formatted Excel file.

## 1. Setup & Configuration

In [ ]:
import requests
import json
import time
import subprocess
import urllib.parse
import os
from collections import defaultdict
from datetime import datetime, timezone
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

DATA_API = "https://data-api.polymarket.com/v1"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept": "application/json",
}
CATEGORIES = ["POLITICS", "ECONOMICS", "FINANCE", "CULTURE", "TECH", "SPORTS", "CRYPTO"]
OUTPUT_PATH = "polymarket_smart_money.xlsx"

session = requests.Session()
session.headers.update(HEADERS)

## 2. API Fetching Functions

In [ ]:
def curl_get(url, params={}):
    if params:
        url = url + "?" + urllib.parse.urlencode(params)
    result = subprocess.run(
        ["curl", "-s", "--http2", "-H", "Accept: application/json",
         "-H", "User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
         "--max-time", "15", url],
        capture_output=True, text=True
    )
    if result.returncode != 0 or not result.stdout.strip():
        return None
    try:
        return json.loads(result.stdout)
    except:
        return None

def fetch_leaderboard(category, period="MONTH", limit=50):
    params = {"timePeriod": period, "limit": limit, "orderBy": "PNL", "category": category}
    try:
        res = session.get(f"{DATA_API}/leaderboard", params=params, timeout=15)
        if res.status_code == 200:
            return res.json()
    except:
        pass
    data = curl_get(f"{DATA_API}/leaderboard", params)
    if data and isinstance(data, list):
        print(f"    curl fallback worked for {category}/{period}: {len(data)} traders")
        return data
    print(f"  Failed: {category}/{period}")
    return []

def fetch_positions(wallet):
    try:
        res = session.get(
            f"{DATA_API}/positions",
            params={"user": wallet, "limit": 100},
            timeout=15
        )
        return res.json() if res.status_code == 200 else []
    except:
        return []

## 3. Data Processing Functions

In [ ]:
def collect_all_leaderboards():
    print("Fetching leaderboards...")
    all_data = {}
    for cat in CATEGORIES:
        print(f"  Category: {cat}")
        all_data[cat] = {
            "MONTH": fetch_leaderboard(cat, "MONTH"),
            "ALL":   fetch_leaderboard(cat, "ALL"),
        }
        time.sleep(0.4)
    return all_data

def build_wallet_map(all_data):
    wallet_map = defaultdict(lambda: {
        "username": "",
        "wallet": "",
        "x_username": "",
        "verified": False,
        "categories_month": [],
        "categories_all": [],
        "pnl_month": 0,
        "vol_month": 0,
        "pnl_all": 0,
        "vol_all": 0,
        "rank_month_best": 999,
        "rank_all_best": 999,
    })

    for cat, periods in all_data.items():
        for trader in periods.get("MONTH", []):
            w = trader.get("proxyWallet", "")
            if not w:
                continue
            wallet_map[w]["wallet"] = w
            wallet_map[w]["username"] = trader.get("userName", w[:10])
            wallet_map[w]["x_username"] = trader.get("xUsername", "")
            wallet_map[w]["verified"] = trader.get("verifiedBadge", False)
            wallet_map[w]["categories_month"].append(cat)
            wallet_map[w]["pnl_month"] = max(wallet_map[w]["pnl_month"], trader.get("pnl", 0))
            wallet_map[w]["vol_month"] = max(wallet_map[w]["vol_month"], trader.get("vol", 0))
            rank = int(trader.get("rank", 999))
            wallet_map[w]["rank_month_best"] = min(wallet_map[w]["rank_month_best"], rank)

        for trader in periods.get("ALL", []):
            w = trader.get("proxyWallet", "")
            if not w:
                continue
            wallet_map[w]["wallet"] = w
            wallet_map[w]["username"] = trader.get("userName", w[:10])
            wallet_map[w]["categories_all"].append(cat)
            wallet_map[w]["pnl_all"] = max(wallet_map[w]["pnl_all"], trader.get("pnl", 0))
            wallet_map[w]["vol_all"] = max(wallet_map[w]["vol_all"], trader.get("vol", 0))
            rank = int(trader.get("rank", 999))
            wallet_map[w]["rank_all_best"] = min(wallet_map[w]["rank_all_best"], rank)

    return wallet_map

def filter_consistent(wallet_map):
    consistent = []
    lucky = []
    for w, info in wallet_map.items():
        cats_month = set(info["categories_month"])
        cats_all   = set(info["categories_all"])
        cross_cat  = len(cats_month)
        in_both = len(cats_month & cats_all)
        info["cross_category_count"] = cross_cat
        info["categories_month_str"] = ", ".join(sorted(cats_month))
        info["categories_all_str"]   = ", ".join(sorted(cats_all))
        info["in_both_periods"]      = in_both > 0
        info["polymarket_url"]       = f"https://polymarket.com/profile/{w}"

        if in_both > 0 or cross_cat >= 3:
            info["tier"] = "CONSISTENT" if in_both > 0 else "DIVERSIFIED"
            if cross_cat >= 3 and in_both > 0:
                info["tier"] = "ELITE"
            consistent.append(info)
        else:
            info["tier"] = "LUCKY"
            lucky.append(info)

    consistent.sort(key=lambda x: (-["LUCKY","DIVERSIFIED","CONSISTENT","ELITE"].index(x["tier"]),
                                    -x["cross_category_count"], -x["pnl_month"]))
    return consistent, lucky

## 4. Excel Generation Function

In [ ]:
def hex_fill(hex_color):
    return PatternFill("solid", start_color=hex_color, fgColor=hex_color)

def build_excel(consistent, lucky, all_data):
    wb = Workbook()

    COLOR_ELITE      = "1A1A2E"
    COLOR_CONSISTENT = "16213E"
    COLOR_HEADER_BG  = "0F3460"
    COLOR_ACCENT     = "E94560"
    COLOR_ELITE_ROW  = "FFF3CD"
    COLOR_CONS_ROW   = "D4EDDA"
    COLOR_DIV_ROW    = "D1ECF1"
    COLOR_LUCKY_ROW  = "F8D7DA"
    COLOR_WHITE      = "FFFFFF"
    COLOR_LIGHT_GRAY = "F8F9FA"

    thin = Side(style="thin", color="CCCCCC")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    def header_style(cell, bg=COLOR_HEADER_BG, font_color=COLOR_WHITE, bold=True, size=10):
        cell.font = Font(name="Arial", bold=bold, color=font_color, size=size)
        cell.fill = hex_fill(bg)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = border

    def data_style(cell, bg=COLOR_WHITE, bold=False, align="left", color="000000"):
        cell.font = Font(name="Arial", bold=bold, color=color, size=9)
        cell.fill = hex_fill(bg)
        cell.alignment = Alignment(horizontal=align, vertical="center", wrap_text=False)
        cell.border = border

    ws1 = wb.active
    ws1.title = "🎯 Consistent Winners"
    ws1.freeze_panes = "A3"
    ws1.row_dimensions[1].height = 30
    ws1.row_dimensions[2].height = 40

    ws1.merge_cells("A1:N1")
    title_cell = ws1["A1"]
    title_cell.value = f"Polymarket Smart Money Tracker — Generated {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M')} UTC"
    title_cell.font = Font(name="Arial", bold=True, size=13, color=COLOR_WHITE)
    title_cell.fill = hex_fill(COLOR_ELITE)
    title_cell.alignment = Alignment(horizontal="center", vertical="center")

    headers = [
        "Tier", "Username", "Wallet Address", "Polymarket Profile",
        "Categories (Month)", "Categories (All Time)", "# Categories",
        "PnL (Month $)", "Volume (Month $)", "PnL (All Time $)",
        "Best Rank (Month)", "Best Rank (All)", "Verified", "X / Twitter"
    ]
    col_widths = [12, 18, 44, 44, 30, 30, 12, 16, 16, 16, 14, 14, 10, 18]

    for col, (h, w) in enumerate(zip(headers, col_widths), 1):
        cell = ws1.cell(row=2, column=col, value=h)
        header_style(cell)
        ws1.column_dimensions[get_column_letter(col)].width = w

    tier_colors = {"ELITE": COLOR_ELITE_ROW, "CONSISTENT": COLOR_CONS_ROW,
                   "DIVERSIFIED": COLOR_DIV_ROW, "LUCKY": COLOR_LUCKY_ROW}
    tier_text   = {"ELITE": "🏆 ELITE", "CONSISTENT": "✅ CONSISTENT",
                   "DIVERSIFIED": "🔀 DIVERSIFIED", "LUCKY": "🎲 LUCKY"}

    for row_idx, info in enumerate(consistent, 3):
        bg = tier_colors.get(info["tier"], COLOR_WHITE)
        row = [
            tier_text.get(info["tier"], info["tier"]),
            info["username"],
            info["wallet"],
            info["polymarket_url"],
            info["categories_month_str"],
            info["categories_all_str"],
            info["cross_category_count"],
            round(info["pnl_month"], 2),
            round(info["vol_month"], 2),
            round(info["pnl_all"], 2),
            info["rank_month_best"] if info["rank_month_best"] < 999 else "-",
            info["rank_all_best"]   if info["rank_all_best"]   < 999 else "-",
            "✓" if info["verified"] else "",
            info["x_username"],
        ]
        for col, val in enumerate(row, 1):
            cell = ws1.cell(row=row_idx, column=col, value=val)
            bold = info["tier"] == "ELITE"
            align = "center" if col in [1, 7, 11, 12, 13] else "left"
            color = COLOR_ACCENT if info["tier"] == "ELITE" and col == 1 else "000000"
            data_style(cell, bg=bg, bold=bold, align=align, color=color)
        ws1.row_dimensions[row_idx].height = 18

    ws2 = wb.create_sheet("🗺️ Cross-Category Map")
    ws2.freeze_panes = "B2"
    ws2.merge_cells(f"A1:{get_column_letter(len(CATEGORIES)+3)}1")
    ws2["A1"].value = "Cross-Category Presence Map — ✓ = appears in top 50 for that category this month"
    ws2["A1"].font = Font(name="Arial", bold=True, size=11, color=COLOR_WHITE)
    ws2["A1"].fill = hex_fill(COLOR_HEADER_BG)
    ws2["A1"].alignment = Alignment(horizontal="center", vertical="center")
    ws2.row_dimensions[1].height = 28

    map_headers = ["Username", "Tier"] + CATEGORIES + ["TOTAL"]
    for col, h in enumerate(map_headers, 1):
        cell = ws2.cell(row=2, column=col, value=h)
        header_style(cell)
        ws2.column_dimensions[get_column_letter(col)].width = 14 if col > 2 else 20

    for row_idx, info in enumerate(consistent, 3):
        bg = tier_colors.get(info["tier"], COLOR_WHITE)
        ws2.cell(row=row_idx, column=1, value=info["username"])
        data_style(ws2.cell(row=row_idx, column=1), bg=bg)
        ws2.cell(row=row_idx, column=2, value=tier_text.get(info["tier"], ""))
        data_style(ws2.cell(row=row_idx, column=2), bg=bg, align="center")

        cats_this_month = set(info["categories_month"])
        for col_idx, cat in enumerate(CATEGORIES, 3):
            present = cat in cats_this_month
            cell = ws2.cell(row=row_idx, column=col_idx, value="✓" if present else "")
            cell_bg = "C3E6CB" if present else bg
            data_style(cell, bg=cell_bg, align="center", bold=present)
        total_cell = ws2.cell(row=row_idx, column=len(CATEGORIES)+3, value=info["cross_category_count"])
        data_style(total_cell, bg=bg, align="center", bold=True)
        ws2.row_dimensions[row_idx].height = 18

    ws3 = wb.create_sheet("📊 Category Leaderboards")
    ws3.freeze_panes = "A3"
    ws3.merge_cells("A1:F1")
    ws3["A1"].value = "Top 50 Per Category — Month"
    ws3["A1"].font = Font(name="Arial", bold=True, size=11, color=COLOR_WHITE)
    ws3["A1"].fill = hex_fill(COLOR_HEADER_BG)
    ws3["A1"].alignment = Alignment(horizontal="center", vertical="center")
    ws3.row_dimensions[1].height = 28

    cat_headers = ["Category", "Rank", "Username", "Wallet", "PnL ($)", "Volume ($)"]
    cat_widths   = [14, 8, 20, 44, 16, 16]
    for col, (h, w) in enumerate(zip(cat_headers, cat_widths), 1):
        cell = ws3.cell(row=2, column=col, value=h)
        header_style(cell)
        ws3.column_dimensions[get_column_letter(col)].width = w

    cat_colors = {
        "POLITICS": "FFF3E0", "ECONOMICS": "E8F5E9", "FINANCE": "E3F2FD",
        "CULTURE": "FCE4EC", "TECH": "EDE7F6", "SPORTS": "E0F7FA", "CRYPTO": "FFFDE7"
    }

    row_idx = 3
    for cat in CATEGORIES:
        traders = all_data.get(cat, {}).get("MONTH", [])
        bg = cat_colors.get(cat, COLOR_WHITE)
        for t in traders:
            row = [
                cat,
                t.get("rank", ""),
                t.get("userName", ""),
                t.get("proxyWallet", ""),
                round(t.get("pnl", 0), 2),
                round(t.get("vol", 0), 2),
            ]
            for col, val in enumerate(row, 1):
                cell = ws3.cell(row=row_idx, column=col, value=val)
                align = "center" if col in [1, 2] else "left"
                data_style(cell, bg=bg, align=align)
            ws3.row_dimensions[row_idx].height = 16
            row_idx += 1

    ws4 = wb.create_sheet("🚨 Copy Trade Watchlist")
    ws4.freeze_panes = "A3"
    ws4.merge_cells("A1:G1")
    ws4["A1"].value = "Copy Trade Watchlist — Elite & Consistent traders to monitor for new positions"
    ws4["A1"].font = Font(name="Arial", bold=True, size=11, color=COLOR_WHITE)
    ws4["A1"].fill = hex_fill(COLOR_ACCENT)
    ws4["A1"].alignment = Alignment(horizontal="center", vertical="center")
    ws4.row_dimensions[1].height = 28

    wl_headers = ["Priority", "Username", "Wallet Address", "Profile URL",
                  "Best Categories", "PnL Month ($)", "Notes"]
    wl_widths   = [10, 18, 44, 44, 30, 16, 30]
    for col, (h, w) in enumerate(zip(wl_headers, wl_widths), 1):
        cell = ws4.cell(row=2, column=col, value=h)
        header_style(cell, bg=COLOR_ACCENT)
        ws4.column_dimensions[get_column_letter(col)].width = w

    priority_map = {"ELITE": "🔴 HIGH", "CONSISTENT": "🟡 MEDIUM", "DIVERSIFIED": "🟢 WATCH"}
    priority_order = ["ELITE", "CONSISTENT", "DIVERSIFIED"]

    row_idx = 3
    for tier in priority_order:
        tier_traders = [x for x in consistent if x["tier"] == tier]
        for info in tier_traders:
            bg = tier_colors.get(tier, COLOR_WHITE)
            row = [
                priority_map.get(tier, ""),
                info["username"],
                info["wallet"],
                info["polymarket_url"],
                info["categories_month_str"],
                round(info["pnl_month"], 2),
                f"Ranks in {info['cross_category_count']} categories" + (" | Verified" if info["verified"] else ""),
            ]
            for col, val in enumerate(row, 1):
                cell = ws4.cell(row=row_idx, column=col, value=val)
                data_style(cell, bg=bg, bold=(tier == "ELITE"), align="left")
            ws4.row_dimensions[row_idx].height = 18
            row_idx += 1

    ws5 = wb.create_sheet("📋 How To Use")
    ws5.column_dimensions["A"].width = 100
    instructions = [
        ("POLYMARKET SMART MONEY TRACKER — GUIDE", True, 14, COLOR_HEADER_BG, COLOR_WHITE),
        ("", False, 10, COLOR_WHITE, "000000"),
        ("SHEET 1 — Consistent Winners", True, 11, "E3F2FD", "000000"),
        ("  All wallets that appear in multiple categories or in BOTH month + all-time top 50.", False, 10, COLOR_WHITE, "000000"),
        ("  ELITE = in multiple categories AND in all-time top 50. These are your best targets.", False, 10, COLOR_WHITE, "000000"),
        ("  CONSISTENT = appears in all-time top 50. Not just a lucky month.", False, 10, COLOR_WHITE, "000000"),
        ("  DIVERSIFIED = ranks in 3+ categories this month. Broad skill.", False, 10, COLOR_WHITE, "000000"),
        ("", False, 10, COLOR_WHITE, "000000"),
        ("SHEET 2 — Cross-Category Map", True, 11, "E3F2FD", "000000"),
        ("  Visual grid showing which categories each trader ranks in. Green = present.", False, 10, COLOR_WHITE, "000000"),
        ("  Traders with the most green cells are most versatile / skillful.", False, 10, COLOR_WHITE, "000000"),
        ("", False, 10, COLOR_WHITE, "000000"),
        ("SHEET 4 — Copy Trade Watchlist", True, 11, "E3F2FD", "000000"),
        ("  RED PRIORITY = monitor these every day. Copy any new position they enter.", False, 10, COLOR_WHITE, "000000"),
        ("  To check new positions: https://data-api.polymarket.com/v1/positions?user=WALLET&limit=100", False, 10, COLOR_WHITE, "000000"),
        ("  To check activity:      https://data-api.polymarket.com/v1/activity?user=WALLET&limit=50", False, 10, COLOR_WHITE, "000000"),
        ("", False, 10, COLOR_WHITE, "000000"),
        ("COPY TRADE WORKFLOW", True, 11, "FFF3CD", "000000"),
        ("  1. Every 5-10 minutes, fetch /positions for all RED PRIORITY wallets", False, 10, COLOR_WHITE, "000000"),
        ("  2. Compare against your saved snapshot (last fetch)", False, 10, COLOR_WHITE, "000000"),
        ("  3. If new position appears → it's a NEW trade they just entered", False, 10, COLOR_WHITE, "000000"),
        ("  4. Log: market, outcome, price, size", False, 10, COLOR_WHITE, "000000"),
        ("  5. Mirror the trade on your own account via CLOB API (needs wallet + API key)", False, 10, COLOR_WHITE, "000000"),
        ("", False, 10, COLOR_WHITE, "000000"),
        ("FILTER LOGIC USED", True, 11, "E8F5E9", "000000"),
        ("  Lucky = only appears in 1 category this month AND not in all-time top 50", False, 10, COLOR_WHITE, "000000"),
        ("  Consistent = appears in all-time top 50 (proves sustained performance)", False, 10, COLOR_WHITE, "000000"),
        ("  Diversified = ranks in 3+ categories (proves broad market knowledge)", False, 10, COLOR_WHITE, "000000"),
        ("  Elite = BOTH consistent AND diversified (highest conviction targets)", False, 10, COLOR_WHITE, "000000"),
    ]

    for row_idx, (text, bold, size, bg, fg) in enumerate(instructions, 1):
        cell = ws5.cell(row=row_idx, column=1, value=text)
        cell.font = Font(name="Arial", bold=bold, size=size, color=fg)
        cell.fill = hex_fill(bg)
        cell.alignment = Alignment(horizontal="left", vertical="center")
        ws5.row_dimensions[row_idx].height = 20

    wb.save(OUTPUT_PATH)
    return OUTPUT_PATH

## 5. Execution

In [ ]:
print("=" * 50)
print("POLYMARKET SMART MONEY TRACKER")
print("=" * 50)

all_data = collect_all_leaderboards()

POLYMARKET SMART MONEY TRACKER
Fetching leaderboards...
  Category: POLITICS
  Category: ECONOMICS
  Category: FINANCE
  Category: CULTURE
  Category: TECH
  Category: SPORTS
  Category: CRYPTO


In [ ]:
print("\nBuilding wallet map...")
wallet_map = build_wallet_map(all_data)
print(f"  Total unique wallets found: {len(wallet_map)}")


Building wallet map...
  Total unique wallets found: 558


In [ ]:
print("Filtering consistent winners...")
consistent, lucky = filter_consistent(wallet_map)
print(f"  Consistent/Elite: {len(consistent)}")
print(f"  Lucky (filtered out): {len(lucky)}")

elite = [x for x in consistent if x["tier"] == "ELITE"]
cons  = [x for x in consistent if x["tier"] == "CONSISTENT"]
div_  = [x for x in consistent if x["tier"] == "DIVERSIFIED"]
print(f"  → ELITE: {len(elite)} | CONSISTENT: {len(cons)} | DIVERSIFIED: {len(div_)}")

Filtering consistent winners...
  Consistent/Elite: 94
  Lucky (filtered out): 464
  → ELITE: 0 | CONSISTENT: 94 | DIVERSIFIED: 0


In [ ]:
print("\nBuilding Excel file...")
output = build_excel(consistent, lucky, all_data)
print(f"\n✅ Done! Saved to: {output}")
print("\nTop ELITE traders:")
for t in elite[:5]:
    print(f"  {t['username']} | PnL: ${t['pnl_month']:,.0f} | Categories: {t['categories_month_str']}")


Building Excel file...


/tmp/ipykernel_7785/3068418876.py:41: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  title_cell.value = f"Polymarket Smart Money Tracker — Generated {datetime.utcnow().strftime('%Y-%m-%d %H:%M')} UTC"



✅ Done! Saved to: polymarket_smart_money.xlsx

Top ELITE traders:
